In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [2]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\train"
VAL_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\val"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\test"

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes

Found 2351 images belonging to 3 classes.
Found 506 images belonging to 3 classes.
Found 505 images belonging to 3 classes.


In [3]:
base_model = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(299, 299, 3)
)

base_model.trainable = False

In [6]:
inputs = layers.Input(shape=(299, 299, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "InceptionV3_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 299, 299, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ inception_v3 (Functional)            │ (None, 8, 8, 2048)          │      21,802,784 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_94               │ (None, 2048)                │           8,192 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22,336,291 (85.21 MB)

 Trainable params: 529,411 (2.02 MB)

 Non-trainable params: 21,806,880 (83.19 MB)

In [7]:
history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5822 - loss: 1.0229
Epoch 1: val_accuracy improved from -inf to 0.84387, saving model to InceptionV3_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.5834 - loss: 1.0208 - val_accuracy: 0.8439 - val_loss: 0.5611 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7697 - loss: 0.6365
Epoch 2: val_accuracy improved from 0.84387 to 0.85178, saving model to InceptionV3_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - accuracy: 0.7698 - loss: 0.6363 - val_accuracy: 0.8518 - val_loss: 0.4388 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8046 - loss: 0.5165
Epoch 3: val_accuracy improved from 0.85178 to 0.86364, saving model to InceptionV3_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 186s 3s/step - accuracy: 0.8046 - loss: 0.5163 - val_accuracy: 0.8636 - val_loss: 0.3711 - learning_rate: 1.0000e-04
Epoch 4/50

In [8]:
import json

history_head_dict = {
    key: [float(x) for x in values]
    for key, values in history_head.history.items()
}

with open("inceptionv3_head_history.json", "w") as f:
    json.dump(history_head_dict, f, indent=4)

In [9]:
model.save("InceptionV3_head._LWDCD.keras")

In [10]:
base_model.trainable = True

for layer in base_model.layers[:249]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8770 - loss: 0.3067
Epoch 1: val_accuracy did not improve from 0.94071
74/74 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.8769 - loss: 0.3069 - val_accuracy: 0.9289 - val_loss: 0.1837 - learning_rate: 1.0000e-05
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8921 - loss: 0.2703
Epoch 2: val_accuracy did not improve from 0.94071
74/74 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.8921 - loss: 0.2703 - val_accuracy: 0.9229 - val_loss: 0.1971 - learning_rate: 1.0000e-05
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8928 - loss: 0.2794
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.

Epoch 3: val_accuracy did not improve from 0.94071
74/74 ━━━━━━━━━━━━━━━━━━━━ 170s 2s/step - accuracy: 0.8927 - loss: 0.2795 - val_accuracy: 0.9190 - val_loss: 0.2037 - learning_rate: 1.0000e-05
Epoch 4/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9030 - loss: 0.2371
Epoch

In [12]:
history_fine_dict = {
    key: [float(x) for x in values]
    for key, values in history_fine.history.items()
}

with open("inceptionv3_finetune_history.json", "w") as f:
    json.dump(history_fine_dict, f, indent=4)

In [13]:
model.save("InceptionV3_finetuned_LWDCD.keras")

In [14]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

16/16 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 0.9091 - loss: 0.2411
Test accuracy: 0.9149


In [15]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step
                  precision    recall  f1-score   support

   Healthy Wheat       0.91      0.89      0.90       175
       Leaf Rust       0.91      0.96      0.94       190
Wheat Loose Smut       0.92      0.89      0.91       140

        accuracy                           0.91       505
       macro avg       0.92      0.91      0.91       505
    weighted avg       0.91      0.91      0.91       505

[[155  10  10]
 [  7 182   1]
 [  8   7 125]]
